# Tayseer: evidence for a phased SAR 40M decision
SDAIA Academy SDA-DSC-112.

**Big Idea:** prioritize regions with substantial adoption gaps and activity, then stage funding to test effectiveness before scaling.

All data are synthetic. The main adoption metric follows the supplied labs: the arithmetic mean of `digital_adoption_pct`. This is an index, not a directly observed digital-transactions share. No causal investment-response model is provided.

[CSV source](https://drive.google.com/file/d/1T6pweFV4EjBxZ75gge775OOYMYx2aKX0/view) · [Dictionary](https://drive.google.com/file/d/1htqVTYconACXDD28CACXOpz8VAyz8Xm5/view) · [Lab 2](https://drive.google.com/file/d/1_DAzbKnnIl4dn1JN2hffwhCueJOoqPOh/view)

Run locally with pandas, numpy and plotly. In Colab, upload the supplied CSV when prompted. Percentage fields already use a 0–100 scale.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
paths=[Path('data/tayseer_services_synthetic.csv'), Path('../data/tayseer_services_synthetic.csv'), Path('tayseer_services_synthetic.csv')]
source=next((p for p in paths if p.exists()), None)
if source is None:
    try:
        from google.colab import files
        uploaded=files.upload()
        source=Path(next(k for k in uploaded if k.endswith('.csv')))
    except ImportError:
        raise FileNotFoundError('Place the course CSV in data/ or beside the notebook.')
df=pd.read_csv(source, parse_dates=['month'])
print('Source:', source, '| shape:', df.shape)

Source: data\tayseer_services_synthetic.csv | shape: (24960, 12)


## 1. Structural quality checks
Check the complete monthly region × service × channel grid. These checks cannot establish real-world validity.

In [2]:
keys=['month','region','service_category','channel']
assert df.shape==(24960,12)
assert not df.isna().any().any()
assert not df.duplicated(keys).any()
assert [df[k].nunique() for k in keys]==[48,13,8,5]
assert df.digital_adoption_pct.between(0,100).all()
assert (df.transactions>0).all()
latest=df.month.max()
current=df.loc[df.month.eq(latest)].copy()
print('No missing cells or duplicate keys; full 48×13×8×5 grid.')
print('Latest month:', latest.strftime('%Y-%m'), '| current rows:',len(current))

No missing cells or duplicate keys; full 48×13×8×5 grid.
Latest month: 2025-12 | current rows: 520


## 2. National trend
Use all 48 months; compare the current month with 65%. Gaps and growth are percentage points.

In [3]:
trend=df.groupby('month',as_index=False).agg(adoption_pct=('digital_adoption_pct','mean'))
national=float(current.digital_adoption_pct.mean())
print(f'January 2022: {trend.adoption_pct.iloc[0]:.6f}%')
print(f'December 2025: {national:.6f}%; gap: {65-national:.6f} pp')
print(f'Growth: {national-trend.adoption_pct.iloc[0]:.6f} pp')
fig=px.line(trend,x='month',y='adoption_pct',title='National course adoption index, Jan 2022–Dec 2025',labels={'adoption_pct':'Adoption index (%)','month':'Month'})
fig.update_yaxes(range=[0,100])
fig.add_hline(y=65,line_dash='dash',line_color='#C25D10',annotation_text='65% target')
fig.update_traces(line_color='#087E87')
fig.show()

January 2022: 51.402423%
December 2025: 63.328346%; gap: 1.671654 pp
Growth: 11.925923 pp


Interactive Plotly chart; trust this notebook to display.

## 3. Regional gaps and a transparent priority proxy
Priority = positive gap / 100 × December transactions. This is not a conversion count or an ROI estimate.

In [4]:
regional=current.groupby('region',as_index=False).agg(adoption_pct=('digital_adoption_pct','mean'),transactions=('transactions','sum'))
regional['gap_pp']=np.maximum(65-regional.adoption_pct,0)
regional['priority_index']=regional.gap_pp/100*regional.transactions
regional=regional.sort_values('priority_index',ascending=False).reset_index(drop=True)
top=regional.head(3).copy()
print('Below target:',int(regional.adoption_pct.lt(65).sum()),'of',len(regional))
print(top.to_string(index=False))
share=100*top.priority_index.sum()/regional.priority_index.sum()
print(f'Top-three share of laggards priority index: {share:.6f}%')
bars=regional.sort_values('adoption_pct')
fig=px.bar(bars,x='adoption_pct',y='region',orientation='h',title='Regional course adoption index, December 2025',labels={'adoption_pct':'Adoption index (%)','region':'Region'})
fig.update_xaxes(range=[0,100])
fig.update_yaxes(categoryorder='array',categoryarray=list(bars.region),autorange='reversed')
fig.add_vline(x=65,line_color='#C25D10',annotation_text='65% target')
fig.show()

Below target: 9 of 13
          region  adoption_pct  transactions  gap_pp  priority_index
          Najran      58.88875          9329 6.11125      570.118512
           Jazan      60.66175         10490 4.33825      455.082425
Northern Borders      60.91225         10917 4.08775      446.259667
Top-three share of laggards priority index: 42.614613%


Interactive Plotly chart; trust this notebook to display.

## 4. Sensitivity: transaction weighting
An alternative definition can reverse national target status and change the third region selected. Agree the metric before scaling; rerank if it changes. Neither index is a verified digital-transactions share.

In [5]:
weighted=current.assign(product=current.digital_adoption_pct*current.transactions).groupby('region').agg(product=('product','sum'),transactions=('transactions','sum'))
regional['weighted_adoption_pct']=regional.region.map(weighted['product']/weighted['transactions'])
regional['weighted_priority_index']=np.maximum(65-regional.weighted_adoption_pct,0)/100*regional.transactions
weighted_national=float(np.average(current.digital_adoption_pct,weights=current.transactions))
weighted_top=regional.sort_values('weighted_priority_index',ascending=False).head(3).region.tolist()
print(f'Weighted national index: {weighted_national:.6f}%')
print('Weighted top three:',weighted_top)
print('Course top three:',top.region.tolist())

Weighted national index: 66.118639%
Weighted top three: ['Najran', 'Northern Borders', 'Al-Baha']
Course top three: ['Najran', 'Jazan', 'Northern Borders']


## 5. Allocation policy
Compare equal distribution, full concentration, and staged concentration. Selected: 35M regional envelope + 5M reserve. The 35M is proportional to priority scores, rounded to whole millions with largest remainders. Amounts are proposals, not quotations. Authorize 10M across the three regions only after approval of metric, baseline, comparison, verified shortlist, service plans and costs, before any intervention; hold 30M. Appoint an owner at approval and launch within 30 days if ready. Day 90 counts from approval: report actual pilot exposure and extend evaluation if insufficient.

In [6]:
ideal=35*(top.priority_index/top.priority_index.sum()).to_numpy()
allocation=np.floor(ideal).astype(int)
remaining=35-int(allocation.sum())
for i in np.argsort(-(ideal-allocation))[:remaining]:
    allocation[i]+=1
top['allocation_sar_m']=allocation
pilot_ideal=top.allocation_sar_m.to_numpy()/35*1000
pilot_units=np.floor(pilot_ideal).astype(int)
for i in np.argsort(-(pilot_ideal-pilot_units))[:1000-int(pilot_units.sum())]:
    pilot_units[i]+=1
top['pilot_sar_m']=pilot_units/100
assert int(top.allocation_sar_m.sum())+5==40
assert round(top.pilot_sar_m.sum(),2)==10
print(top[['region','allocation_sar_m','pilot_sar_m']].to_string(index=False))
print('Reserve: 5M; total envelope: 40M; initial release: 10M; held: 30M.')

          region  allocation_sar_m  pilot_sar_m
          Najran                13         3.72
           Jazan                11         3.14
Northern Borders                11         3.14
Reserve: 5M; total envelope: 40M; initial release: 10M; held: 30M.


## 6. Conditional effect and review
If the three selected regions each reach 65% and the other ten stay unchanged, calculate the national arithmetic-mean index. This is conditional arithmetic, not an investment forecast. A proposed day-90 continuation goal is +2pp per pilot with stable CSAT/SLA, an appropriate comparison and an accepted cost. Validate baseline, seasonality and service mix. Candidate onboarding/friction interventions remain hypotheses.

In [7]:
scenario=national+top.gap_pp.sum()/len(regional)
print(f'Conditional national index: {scenario:.6f}%')
print(f'Remaining target gap: {65-scenario:.6f} pp')
# Reconcile independently calculated notebook results with the packaged evidence.
import json
summary_path=next((p for p in [Path('analysis/summary.json'),Path('summary.json')] if p.exists()),None)
if summary_path:
    expected=json.loads(summary_path.read_text(encoding='utf-8'))
    assert abs(national-expected['national_latest_pct'])<1e-10
    assert top.region.tolist()==expected['selected_regions']
    assert abs(share-expected['selected_priority_index_share_pct'])<1e-10
    assert weighted_top==expected['weighted_sensitivity_top3']
    assert abs(scenario-expected['national_if_selected_reach_65_pct'])<1e-10
    assert top.allocation_sar_m.tolist()==[r['allocation_sar_m'] for r in expected['selected_allocations']]
    print('Notebook reconciles with packaged summary and allocations.')

Conditional national index: 64.446596%
Remaining target gap: 0.553404 pp
Notebook reconciles with packaged summary and allocations.


## Delivery
Use the seven slides as the primary story. Orient the audience before interpreting charts. Use live Tableau for supplementary evidence and Q&A after actual opening verification. Keep the presentation within seven minutes; ask in the first 30 seconds and again at closing. Do not sum unique users across services/channels because individuals may overlap.